©2026. For information, contact Deloitte Tohmatsu Group.

# 📝 演習概要

この演習では、ニューラルネットワークで用いられる主要な活性化関数をPythonで実装・可視化します。ステップ関数、ReLU、シグモイド、tanh、ソフトマックスの各関数について、数式に基づくコード実装とグラフ描画を行い、それぞれの特性や用途の違いを視覚的に理解します。

# 事前準備

[JDLAが策定しているバージョン](https://www.jdla.org/certificate/engineer/)に合わせるために、以下のセルの実行をお願いします．

（#コメントアウト されているものは必要ありません）

また実行完了後に「ランタイムの再起動」をして下さい．

（以下のセルの実行は、最初にしていただければ、以降必要ありません．）


In [ ]:
%%capture
!pip uninstall matplotlib -y
!pip install matplotlib==3.9.4

# !pip uninstall opencv-python -y
# !pip install opencv-python==4.11.0.86

# !pip uninstall torch -y
# !pip install torch==2.7.0

# !pip uninstall torchvision -y
# !pip install torchvision==0.22.0

# 活性化関数

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

!pip install japanize_matplotlib
import japanize_matplotlib  # Matplotlibの日本語フォント崩れ対策

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 65.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for japanize_matplotlib: filename=japanize_matplotlib-1.1.3-py3-none-any.whl size=4120257 sha256=40f6ac7cfdb049853d872edf972a1f4108a2733e45b476e63280323e7620238b
  Stored in directory: /root/.cache/pip/wheels/c1/f7/9b/418f19a7b9340fc16e071e89efc379aca68d40238b258df53d
Successfully built japanize_matplotlib


## ステップ関数

In [ ]:
# ステップ関数（Heaviside step function）
#   - 閾値thetaより大きければ1、小さければ0を返す

def step_function(x, theta):
    # x > theta のとき True(=1), それ以外 False(=0) を返す
  return np.array(x>theta)

# -3 から 3 までを0.05刻みでサンプル
x = np.arange(-3.0, 3.0, 0.05)
y = step_function(x, theta=0.5)
plt.title('ステップ関数')
plt.plot(x, y)
plt.ylim(-1.5, 3)
plt.grid(True)
plt.show()

## ReLU関数

In [ ]:
# ReLU関数 (Rectified Linear Unit)
#   - x>0 のとき x, それ以外は0

def relu_function(x):
  return np.maximum(0, x)

x = np.arange(-3.0, 3.0, 0.05)
y = relu_function(x)
plt.title('ReLU関数')
plt.plot(x, y)
plt.ylim(-1.5, 3)
plt.grid(True)
plt.show()

In [ ]:
# Relu関数と微分
#   relu(x)        = max(0, x)
#   relu'(x)       = 1 (x>=0のとき), 0 (それ以外)

def relu(x):
    return np.maximum(0, x)

def deriv_relu(x):
    grad = np.zeros_like(x)
    grad[x>=0] = 1  # x>=0 なら勾配1、それ以外0
    return grad

In [ ]:
x = np.arange(-3.0, 3.0, 0.05)
y = relu_function(x)
dy = deriv_relu(x)
plt.title('ReLU関数と微分')
plt.plot(x, y)
plt.plot(x, dy)
plt.ylim(-1.5, 3)
plt.grid(True)
plt.show()

In [ ]:
# Leakey Relu関数
#   - ReLUは x<0 のとき完全に0になってしまうのでニューロンが死ぬことがある
#   - Leaky ReLUは x<0 のときも 0.1x のような小さい負の傾きで勾配を残す
#   - そのおかげで「負側で勾配0にならない」→学習が止まりにくい

def LRelu(x):
    y = np.where(x > 0, x , 0.1 * x)
    return y

def deriv_LRelu(X):
    y = np.where(x>0,1, 0.1)
    return y

In [ ]:
x = np.arange(-3.0, 3.0, 0.05)
y = LRelu(x)
dy = deriv_LRelu(x)
plt.title('Leaky_ReLU関数と微分')
plt.plot(x, y)
plt.plot(x, dy)
plt.ylim(-0.5, 1.5)
plt.grid(True)
plt.show()

In [ ]:
# ELU (Exponential Linear Unit)
#   - x>0 はそのまま線形（= ReLUと同じ）
#   - x<=0 では指数関数的に滑らかに下がる
#   - ReLUよりも「なめらか」、勾配が0になりにくい

def elu(x, alpha=1.0):
    """
    ELU:
      x > 0  のとき x
      x <= 0 のとき alpha*(exp(x)-1)
    alpha は負側の飽和領域の深さを決める係数
    """
    return np.where(x > 0.0, x, alpha * (np.exp(x) - 1))

def deriv_elu(x, alpha=1.0):
    """
    ELUの微分:
      x > 0  のとき 1
      x <= 0 のとき ELU(x) + alpha
      （ELUの性質からそうなる）
    """
    return np.where(x > 0.0, 1.0, elu(x, alpha) + alpha)

In [ ]:
x = np.arange(-3.0, 3.0, 0.05)
y = elu(x)
dy = deriv_elu(x)
plt.title('ELU関数と微分')
plt.plot(x, y)
plt.plot(x, dy)
plt.ylim(-1.0, 3.0)
plt.grid(True)
plt.show()

## シグモイド関数

In [ ]:
# シグモイド関数 (logistic sigmoid)
#   - [0,1] の範囲に押し込むS字カーブ
#   - 古典的な活性化関数（出力を確率っぽく解釈できる）
#   - ただし勾配が小さくなりやすい(=勾配消失しやすい)

def sigmoid_function(x):
  return 1 /( 1 + np.exp(-x))

x = np.arange(-5, 5, 0.1)
y = sigmoid_function(x)
plt.plot(x, y)
plt.ylim(0.0, 1)
plt.title('シグモイド関数')
plt.grid(True)
plt.show()

In [ ]:
# シグモイド関数と微分
#   sigmoid(x) = 1 / (1 + e^-x)
#   sigmoid'(x)= sigmoid(x) * (1 - sigmoid(x))
#   出力が0や1に近づくと勾配が0に近くなる → 深い層だと勾配が消えて困る

def sigmoid(x):
    return 1/(1 + np.exp(-x))

def deriv_sigmoid(x):
    return (1 - sigmoid(x)) * sigmoid(x)

In [ ]:
x = np.arange(-3.0, 3.0, 0.05)
y = sigmoid(x)
dy = deriv_sigmoid(x)
plt.title('Sigmoid関数と微分')
plt.plot(x, y)
plt.plot(x, dy)
plt.ylim(-0.5, 1.5)
plt.grid(True)
plt.show()

## tanh関数

In [ ]:
# tanh 関数（双曲線正接）
#   - 出力レンジは [-1, 1]
#   - Sigmoidよりも出力の平均が0に近いので学習がやや安定しやすい

def tanh_function(x):
  return (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))

x = np.arange(-3, 3, 0.1)
y = tanh_function(x)
plt.plot(x, y)
plt.ylim(-1.5, 1.5)
plt.title('Tanh関数')
plt.grid(True)
plt.show()

In [ ]:
# tanh 関数（双曲線正接）
#   tanh'(x) = 1 - tanh(x)^2
#   シグモイド同様に飽和領域では勾配が小さい(=勾配消失しやすい)

def tanh_function(x):
  return (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))

def deriv_tanh(x):
  return 1 - tanh_function(x) ** 2

In [ ]:
x = np.arange(-3.0, 3.0, 0.05)
y = tanh_function(x)
dy = deriv_tanh(x)
plt.title('Tanh関数と微分')
plt.plot(x, y)
plt.plot(x, dy)
plt.ylim(-1.5, 1.5)
plt.grid(True)
plt.show()

## ソフトマックス関数

In [ ]:
# Softmax関数
#   - ベクトル a = [a1, a2, ..., aK] を
#     確率分布っぽい形 [p1, p2, ..., pK] に正規化する
#
#   数値安定性のために max_a = max(a) を引いてから exp している。
#   これをしないと exp() がオーバーフローして inf になることがある。

def softmax(a):
    max_a = np.max(a, axis=-1, keepdims=True)  # 各サンプルごとに最大値を引いて安定化
    exp_a = np.exp(a - max_a)
    sum_exp_a = np.sum(exp_a, axis=-1, keepdims=True)
    y = exp_a / sum_exp_a  # 各成分を総和で割って確率にする

    return y